#### Loading the Consolidated Passport

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Load the time-series data
ts_df = pd.read_csv('../data/processed/ndvi_baseline_2019_2025.csv')
ts_df['observation_date'] = pd.to_datetime(ts_df['observation_date'])

print(f"Loaded {len(ts_df)} months of vegetation history for deep trend analysis.")

Loaded 84 months of vegetation history for deep trend analysis.


#### Calculating the Stability Score

In [2]:
# Calculate mean and standard deviation of NDVI
ndvi_mean = ts_df['ndvi_value'].mean()
ndvi_std = ts_df['ndvi_value'].std()

# Coefficient of Variation (Standard Industry Metric for Stability)
# Lower is better (< 10% is excellent for agroforestry)
cv = (ndvi_std / ndvi_mean) * 100

print(f"Baseline Stability (CV): {cv:.2f}%")

if cv < 10:
    stability_rating = "High Stability (Consolidated Carbon)"
else:
    stability_rating = "Variable (High Risk of Disturbance)"

print(f"Audit Rating: {stability_rating}")

Baseline Stability (CV): 4.61%
Audit Rating: High Stability (Consolidated Carbon)


#### Anomaly Detection (Z-Score Analysis)

In [3]:
# Calculate Z-Scores for each observation
ts_df['z_score'] = (ts_df['ndvi_value'] - ndvi_mean) / ndvi_std

# Identify Anomaly: Any month 2 standard deviations below the mean
anomalies = ts_df[ts_df['z_score'] < -2]

if not anomalies.empty:
    print(f"Warning: {len(anomalies)} vegetation anomalies detected.")
    print(anomalies[['observation_date', 'ndvi_value', 'z_score']])
else:
    print("No significant vegetation anomalies detected (2019-2025).")

No significant vegetation anomalies detected (2019-2025).


#### Linear Trend Projection (Slope)

In [5]:
import datetime

# Use ordinal dates for linear regression
ts_df['date_ordinal'] = ts_df['observation_date'].map(datetime.datetime.toordinal)

slope, intercept, r_value, p_value, std_err = stats.linregress(ts_df['date_ordinal'], ts_df['ndvi_value'])

# Annual Change Rate (Slope * 365 days)
annual_change = slope * 365

print(f"Long-term Trend: {annual_change:+.4f} NDVI units / year")
print(f"R-squared: {r_value**2:.4f}")

if annual_change > 0:
    trend_desc = "Increasing Biomass Density"
else:
    trend_desc = "Gradual Thinning Detected"

print(f"Interpretation: {trend_desc}")

Long-term Trend: +0.0021 NDVI units / year
R-squared: 0.0134
Interpretation: Increasing Biomass Density
